# 02 - Forecasting

Baselines and models for `aggregate_cpu_utilization`, evaluated with rolling-origin
validation **inside the 110-point training window**. The 24-point holdout is used
only once, for the final evaluation.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from cwfc.targets import build_targets, split_train_holdout
from cwfc.validation import rolling_origin_folds
from cwfc.baselines import persistence_forecast, moving_average_forecast
from cwfc.metrics import mae, rmse
from cwfc.models import fit_ar1, ar1_forecast

DATA_DIR = Path("../challenge_data")
TARGET = "aggregate_cpu_utilization"
HORIZON = 24

cpu = pd.read_csv(DATA_DIR / "vm_cpu_readings.csv.gz")
targets = build_targets(cpu)
train, _ = split_train_holdout(targets, HORIZON)
origins = rolling_origin_folds(len(train), horizon=HORIZON)

print(origins)

[50, 56, 62, 68, 74, 80, 86]


## Validation design

Rolling-origin evaluation on the training window: horizon 24 (same as the holdout),
minimum 48 points of history, step 6 (30 min), origins anchored at the end so the last fold ends at point 109. This gives 7 folds. Each fold mimics production:
the model sees only data before the origin and forecasts the next 2 hours.
Folds starting at 50-62 include the elevated episode at ~4.5-6 h, so models are also tested on a level shift.

In [2]:
methods = {
    "persistence":  lambda tr, te: persistence_forecast(tr, te, TARGET),
    "ma_6":         lambda tr, te: moving_average_forecast(tr, te, TARGET, 6),
    "ma_12":        lambda tr, te: moving_average_forecast(tr, te, TARGET, 12),
    "train_mean":   lambda tr, te: moving_average_forecast(tr, te, TARGET, len(tr)),
    "ar1": lambda tr, te: ar1_forecast(tr, te, TARGET),
}

rows = []
for origin in origins:
    tr = train.iloc[:origin]
    te = train.iloc[origin:origin + HORIZON]
    for name, fn in methods.items():
        pred = fn(tr, te)
        rows.append({"origin": origin, "method": name, "mae": mae(te[TARGET], pred), "rmse": rmse(te[TARGET], pred)})

results = pd.DataFrame(rows)
results.groupby("method")[["mae", "rmse"]].mean().sort_values("mae").round(4)

,mae,rmse
method,,
ar1,0.2009,0.2300
train_mean,0.2088,0.2384
persistence,0.2152,0.2514
ma_6,0.2428,0.2724
ma_12,0.2595,0.2882


In [3]:
results.pivot(index="origin", columns="method", values="mae").round(3)

method,ar1,ma_12,ma_6,persistence,train_mean
origin,,,,,
50,0.303,0.324,0.316,0.268,0.303
56,0.265,0.242,0.198,0.296,0.271
62,0.150,0.220,0.312,0.297,0.173
68,0.152,0.390,0.353,0.301,0.164
74,0.168,0.355,0.296,0.124,0.171
80,0.165,0.167,0.085,0.087,0.170
86,0.204,0.119,0.140,0.132,0.210


## Baseline results

| method | mean MAE | mean RMSE |
|---|---|---|
| train_mean | 0.209 | 0.238 |
| persistence | 0.215 | 0.251 |
| ma_6 | 0.243 | 0.272 |
| ma_12 | 0.260 | 0.288 |

- MAE ≈ 0.21 pp on a ~9.4% level (~2% relative error).
- **No single winner across folds.** `train_mean` wins when the series reverts after
the elevated episode (origins 62, 68); `persistence` wins in calm periods (74, 80).
The 3% gap in mean MAE between them is within fold-to-fold variation.
- Short moving averages are rarely best: they carry recent deviations forward.
- Best MAE (0.21) is close to the series std (0.19): most variation is
short-term noise around a stable level.

**Next:** a mean-reverting model (AR(1)) that starts from the last value and
decays towards the mean combines the strengths of both baselines.

In [4]:
mu, phi = fit_ar1(train[TARGET])
print(f"mu = {mu:.3f}, phi={phi:.3f}")
print("phi^h for h = 1, 6, 12, 24:", [round(phi ** h, 3) for h in (1, 6, 12, 24)])

mu = 9.414, phi=0.672
phi^h for h = 1, 6, 12, 24: [0.672, 0.092, 0.009, 0.0]


## AR(1) model

Fitted on the full training window: mu = 9.414, phi = 0.672.
phi^h = 0.67, 0.09, 0.01, 0.00 for h = 1, 6, 12, 24: deviations from the mean
decay within ~30 minutes, so the 2-hour forecast quickly converges to the mean.

| method | mean MAE | mean RMSE | MAE vs persistence |
|---|---|---|---|
| ar1 | 0.201 | 0.230 | −6.6% |
| train_mean | 0.209 | 0.238 | −3.0% |
| persistence | 0.215 | 0.251 | — |

- AR(1) beats `train_mean` in 6 of 7 folds (one tie): a **consistent** gain.
- AR(1) beats `persistence` in 3 of 7 folds (56, 62, 68: the elevated episode
and its reversion); persistence is better in calm periods (74, 80, 86).
- AR(1) is **never the worst** method in any fold, which matters more for
production than winning individual folds.
- The gain is modest, consistent with most variation being unpredictable noise.

I select **AR(1)** as the main model for `aggregate_cpu_utilization`.
For `active_vm_count` (constant) AR(1) reduces to the mean (phi = 0) = 10 000.